In [1]:
import camelot

# Read all tables from PDF
tables = camelot.read_pdf("../datasets/Market Watch-96.pdf", pages="all", flavor='stream')  # 'stream' works better for aligned text

# Check number of tables extracted
print(f"Total tables found: {len(tables)}")

# Preview first table
tables[0].df.head()

d:\Smart_Agri_System\venv\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4
d:\Smart_Agri_System\venv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (49.544, 748.25928, 582.8084, 866.481460841368)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


Total tables found: 4


,0
0,Jointly produced by:
1,"WFP-World Food Programme, Evidence, Policy and"
2,"Innovation Unit, Nepal"
3,MoALD-Ministry of Agriculture and Livestock De...
4,"(MoALD), Department of Agriculture (DoA), Market"


In [2]:
import pandas as pd

all_tables = [table.df for table in tables]  # list of dataframes
df = pd.concat(all_tables, ignore_index=True)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,Jointly produced by:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"WFP-World Food Programme, Evidence, Policy and",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"Innovation Unit, Nepal",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MoALD-Ministry of Agriculture and Livestock De...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"(MoALD), Department of Agriculture (DoA), Market",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Rename columns to match your format
df.columns = ['date', 'crop', 'market', 'price']

# Remove empty rows
df = df.dropna(how='any')

# Convert price to numeric
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Drop rows with invalid date/price
df = df.dropna(subset=['date','price'])

ValueError: Length mismatch: Expected axis has 18 elements, new values have 4 elements

In [4]:
import camelot

tables = camelot.read_pdf("../datasets/Market Watch-96.pdf", pages="all", flavor='stream')
print(f"Total tables found: {len(tables)}")

# Preview each table to find the real data table
for i, table in enumerate(tables):
    print(f"\nTable {i} preview:")
    print(table.df.head())

d:\Smart_Agri_System\venv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (49.544, 748.25928, 582.8084, 866.481460841368)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


Total tables found: 4

Table 0 preview:
                                                   0
0                               Jointly produced by:
1     WFP-World Food Programme, Evidence, Policy and
2                             Innovation Unit, Nepal
3  MoALD-Ministry of Agriculture and Livestock De...
4   (MoALD), Department of Agriculture (DoA), Market

Table 1 preview:
                                                   0
0                                     February 2020†
1                                         HIGHLIGHTS
2  In  February  2020,  retail  prices  of  most ...
3  remained  relatively  stable  or  experienced ...
4          commodities and an overall smooth supply.

Table 2 preview:
       0          1        2  3                     4  5  6   \
0                     Current                                  
1                                                              
2                       price     Change in price (%)*         
3          Commodity     (NRS   

In [5]:
# Pick the likely table
df = tables[2].df  # Table 2 seems like the actual data
df.head(10)         # Inspect the first 10 rows to see headers

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,,,Current,,,,,,,,,,,,,,,
1,,,,,,,,,,,,Current,,,,,,
2,,,price,,Change in price (%)*,,,Average change over,,,,,,Change in price (%)*,,,Average change over,
3,,Commodity,(NRS,,,,,,,,,price,,,,,,
4,Market,,,,,,,,,Market,Commodity,,,,,,,
5,,,per Kg/,,,,,,,,,(NRS per,,,,,,
6,,,,,,,,,,,,Kg/Lt.),,,,,,
7,,,Lt.),1 m,3 m,1 yr,,1m 3m 1yr,,,,,1 m,3 m,1 yr,,1m 3m 1yr,
8,,Major consumer markets,,,,,,,,,,,,,,,,
9,,,,,,,,,,,Coarse rice,42.0,5.0,20.0,13.5,►,▲,►


In [6]:
# Assume tables[2] contains this data
df = tables[2].df

# Skip first 9 rows of headers
df = df[9:].reset_index(drop=True)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,,,,,,,,,,,Coarse rice,42.0,5.0,20.0,13.5,►,▲,►
1,,Coarse rice,47.0,0.0,0.0,0.0,►,►,►,,,,,,,,,
2,,,,,,,,,,,Wheat flour,46.0,0.0,4.5,4.5,►,►,►
3,,,60.0,0.0,0.0,25.0,►,►,▲,,,,,,,,,
4,,Wheat flour,,,,,,,,,Soybean oil,147.0,0.0,11.4,7.3,►,▲,►


In [7]:
# Adjust based on your column index
df = df[[1, 2, 3, 4]]  # crop + 3 markets
df.columns = ['crop', 'Kathmandu', 'Biratnagar', 'Hetauda']
df.head()

,crop,Kathmandu,Biratnagar,Hetauda
0,,,,
1,Coarse rice,47.0,0.0,0.0
2,,,,
3,,60.0,0.0,0.0
4,Wheat flour,,,


In [8]:
df_long = df.melt(id_vars=['crop'], 
                  value_vars=['Kathmandu', 'Biratnagar', 'Hetauda'], 
                  var_name='market', 
                  value_name='price')

# Convert price to numeric
df_long['price'] = pd.to_numeric(df_long['price'], errors='coerce')

# Drop rows with missing prices
df_long = df_long.dropna(subset=['price'])

In [9]:
df_long['date'] = pd.to_datetime('2020-02-01')
df_long = df_long[['date','crop','market','price']]
df_long.head()

,date,crop,market,price
1,2020-02-01,Coarse rice,Kathmandu,47.0
3,2020-02-01,,Kathmandu,60.0
5,2020-02-01,Soybean oil,Kathmandu,140.0
7,2020-02-01,Chicken,Kathmandu,400.0
9,2020-02-01,Broken lentil,Kathmandu,110.0


In [10]:
df_long.to_csv("../datasets/market_prices_feb2020.csv", index=False)